In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_allscripts.provider;

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_tw.provider;

In [0]:
%sql
-- DELETE FROM _exponent.omop_silver.provider
-- WHERE source_system = 'allscripts_tw';

In [0]:
%sql
-- DELETE FROM _exponent.omop_mapping.source_to_provider
-- WHERE source_system = 'allscripts_tw';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver AS
SELECT
dbo_person.fullname AS provider_name,
dbo_provider.npi AS npi,
dbo_provider.deanumber AS dea,
COALESCE(domain_source_to_concept.omop_concept_id, 0) AS specialty_concept_id,
NULL AS care_site_id,
YEAR(dbo_person.dateofbirth) AS year_of_birth,
COALESCE(gender_concept.omop_concept_id, 0) AS gender_concept_id,
CONCAT_WS(CHR(31), 'allscripts_tw','dbo_provider', 'id', CAST(dbo_provider.id AS BIGINT)) AS provider_source_value,
dbo_specialty_de.entryname AS specialty_source_value,
0 AS specialty_source_concept_id,
dbo_sex_de.entryname AS gender_source_value,
0 AS gender_source_concept_id,
'allscripts_tw' AS source_system
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_provider
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person
ON dbo_person.id = dbo_provider.id
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_sex_de
ON dbo_sex_de.id = dbo_person.sexde
LEFT OUTER JOIN _exponent.omop_mapping.domain_source_to_concept gender_concept
ON gender_concept.source_id = dbo_person.sexde
AND gender_concept.domain_id = 'Gender'
AND gender_concept.source_system = 'allscripts_tw'
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_specialty_de 
ON dbo_specialty_de.id = dbo_provider.specialtyde
LEFT OUTER JOIN _exponent.omop_mapping.domain_source_to_concept
ON domain_source_to_concept.source_id = dbo_specialty_de.id
AND domain_source_to_concept.domain_id = 'Provider'
AND domain_source_to_concept.source_system = 'allscripts_tw'
WHERE 1=1
-- AND dbo_person.etl_load_ts BETWEEN  CURRENT_DATE() - INTERVAL 365 DAY AND CURRENT_DATE()

In [0]:
%sql
MERGE INTO _exponent.omop_silver.provider AS target
USING silver AS source
ON target.provider_source_value = source.provider_source_value

WHEN MATCHED AND NOT (
     target.provider_name               <=> source.provider_name
 AND target.npi                         <=> source.npi
 AND target.dea                         <=> source.dea
 AND target.specialty_concept_id        <=> source.specialty_concept_id
 AND target.care_site_id                <=> source.care_site_id
 AND target.year_of_birth               <=> source.year_of_birth
 AND target.gender_concept_id           <=> source.gender_concept_id
 AND target.specialty_source_value      <=> source.specialty_source_value
 AND target.specialty_source_concept_id <=> source.specialty_source_concept_id
 AND target.gender_source_value         <=> source.gender_source_value
 AND target.gender_source_concept_id    <=> source.gender_source_concept_id
 AND target.source_system               <=> source.source_system
)
THEN UPDATE SET
  target.provider_name               = source.provider_name,
  target.npi                         = source.npi,
  target.dea                         = source.dea,
  target.specialty_concept_id        = source.specialty_concept_id,
  target.care_site_id                = source.care_site_id,
  target.year_of_birth               = source.year_of_birth,
  target.gender_concept_id           = source.gender_concept_id,
  target.specialty_source_value      = source.specialty_source_value,
  target.specialty_source_concept_id = source.specialty_source_concept_id,
  target.gender_source_value         = source.gender_source_value,
  target.gender_source_concept_id    = source.gender_source_concept_id,
  target.source_system               = source.source_system,
  target.last_mod_tsp                = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  provider_name,
  npi,
  dea,
  specialty_concept_id,
  care_site_id,
  year_of_birth,
  gender_concept_id,
  provider_source_value,
  specialty_source_value,
  specialty_source_concept_id,
  gender_source_value,
  gender_source_concept_id,
  source_system,
  last_mod_tsp
) VALUES (
  source.provider_name,
  source.npi,
  source.dea,
  source.specialty_concept_id,
  source.care_site_id,
  source.year_of_birth,
  source.gender_concept_id,
  source.provider_source_value,
  source.specialty_source_value,
  source.specialty_source_concept_id,
  source.gender_source_value,
  source.gender_source_concept_id,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_provider (
    source_system,
    provider_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.provider_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        provider_source_value,
        last_mod_tsp
    FROM _exponent.omop_silver.provider
    WHERE provider_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_provider x
  ON s.provider_source_value = x.provider_source_value;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold AS
SELECT
  source_to_provider.provider_id,
  provider.provider_name,
  provider.npi,
  provider.dea,
  provider.specialty_concept_id,
  provider.care_site_id,
  provider.year_of_birth,
  provider.gender_concept_id,
  provider.provider_source_value,
  provider.specialty_source_value,
  provider.specialty_source_concept_id,
  provider.gender_source_value,
  provider.gender_source_concept_id,
  provider.last_mod_tsp
FROM _exponent.omop_silver.provider
JOIN _exponent.omop_mapping.source_to_provider
  ON provider.provider_source_value = source_to_provider.provider_source_value
 AND source_to_provider.active_flag = true
 WHERE 1=1
 AND provider.source_system = 'allscripts_tw';


In [0]:
%sql
MERGE INTO _exponent.omop_tw.provider AS target
USING gold AS source
ON target.provider_id = source.provider_id

WHEN MATCHED AND NOT (
     target.provider_name               <=> source.provider_name
 AND target.npi                         <=> source.npi
 AND target.dea                         <=> source.dea
 AND target.specialty_concept_id        <=> source.specialty_concept_id
 AND target.care_site_id                <=> source.care_site_id
 AND target.year_of_birth               <=> source.year_of_birth
 AND target.gender_concept_id           <=> source.gender_concept_id
 AND target.provider_source_value       <=> source.provider_source_value
 AND target.specialty_source_value      <=> source.specialty_source_value
 AND target.specialty_source_concept_id <=> source.specialty_source_concept_id
 AND target.gender_source_value         <=> source.gender_source_value
 AND target.gender_source_concept_id    <=> source.gender_source_concept_id
)
THEN UPDATE SET
  target.provider_name               = source.provider_name,
  target.npi                         = source.npi,
  target.dea                         = source.dea,
  target.specialty_concept_id        = source.specialty_concept_id,
  target.care_site_id                = source.care_site_id,
  target.year_of_birth               = source.year_of_birth,
  target.gender_concept_id           = source.gender_concept_id,
  target.provider_source_value       = source.provider_source_value,
  target.specialty_source_value      = source.specialty_source_value,
  target.specialty_source_concept_id = source.specialty_source_concept_id,
  target.gender_source_value         = source.gender_source_value,
  target.gender_source_concept_id    = source.gender_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  provider_id,
  provider_name,
  npi,
  dea,
  specialty_concept_id,
  care_site_id,
  year_of_birth,
  gender_concept_id,
  provider_source_value,
  specialty_source_value,
  specialty_source_concept_id,
  gender_source_value,
  gender_source_concept_id
) VALUES (
  source.provider_id,
  source.provider_name,
  source.npi,
  source.dea,
  source.specialty_concept_id,
  source.care_site_id,
  source.year_of_birth,
  source.gender_concept_id,
  source.provider_source_value,
  source.specialty_source_value,
  source.specialty_source_concept_id,
  source.gender_source_value,
  source.gender_source_concept_id
);


In [0]:
%sql
MERGE INTO _exponent.omop_allscripts.provider AS target
USING gold AS source
ON target.provider_id = source.provider_id

WHEN MATCHED AND NOT (
     target.provider_name               <=> source.provider_name
 AND target.npi                         <=> source.npi
 AND target.dea                         <=> source.dea
 AND target.specialty_concept_id        <=> source.specialty_concept_id
 AND target.care_site_id                <=> source.care_site_id
 AND target.year_of_birth               <=> source.year_of_birth
 AND target.gender_concept_id           <=> source.gender_concept_id
 AND target.provider_source_value       <=> source.provider_source_value
 AND target.specialty_source_value      <=> source.specialty_source_value
 AND target.specialty_source_concept_id <=> source.specialty_source_concept_id
 AND target.gender_source_value         <=> source.gender_source_value
 AND target.gender_source_concept_id    <=> source.gender_source_concept_id
)
THEN UPDATE SET
  target.provider_name               = source.provider_name,
  target.npi                         = source.npi,
  target.dea                         = source.dea,
  target.specialty_concept_id        = source.specialty_concept_id,
  target.care_site_id                = source.care_site_id,
  target.year_of_birth               = source.year_of_birth,
  target.gender_concept_id           = source.gender_concept_id,
  target.provider_source_value       = source.provider_source_value,
  target.specialty_source_value      = source.specialty_source_value,
  target.specialty_source_concept_id = source.specialty_source_concept_id,
  target.gender_source_value         = source.gender_source_value,
  target.gender_source_concept_id    = source.gender_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  provider_id,
  provider_name,
  npi,
  dea,
  specialty_concept_id,
  care_site_id,
  year_of_birth,
  gender_concept_id,
  provider_source_value,
  specialty_source_value,
  specialty_source_concept_id,
  gender_source_value,
  gender_source_concept_id
) VALUES (
  source.provider_id,
  source.provider_name,
  source.npi,
  source.dea,
  source.specialty_concept_id,
  source.care_site_id,
  source.year_of_birth,
  source.gender_concept_id,
  source.provider_source_value,
  source.specialty_source_value,
  source.specialty_source_concept_id,
  source.gender_source_value,
  source.gender_source_concept_id
);
